# 00. Imports & Upload DB

In [36]:
# Машинное обучение и метрики
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    classification_report,
    balanced_accuracy_score
)
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier
import torch
from tqdm import tqdm


# LightAutoML incl report generation
from lightautoml.automl.presets.tabular_presets import TabularAutoML, TabularUtilizedAutoML
from lightautoml.dataset.roles import DatetimeRole
from lightautoml.tasks import Task
#from lightautoml.report.report_deco import ReportDeco, ReportDecoUtilized
from lightautoml.addons.tabular_interpretation import SSWARM

# LightAutoML time-based cross-validation
from lightautoml.validation.np_iterators import TimeSeriesIterator

from lightautoml.reader.guess_roles import gini_normalized


'nlp' extra dependency package 'gensim' isn't installed. Look at README.md in repo 'LightAutoML' for installation instructions.
'nlp' extra dependency package 'nltk' isn't installed. Look at README.md in repo 'LightAutoML' for installation instructions.
'nlp' extra dependency package 'transformers' isn't installed. Look at README.md in repo 'LightAutoML' for installation instructions.
'nlp' extra dependency package 'gensim' isn't installed. Look at README.md in repo 'LightAutoML' for installation instructions.
'nlp' extra dependency package 'nltk' isn't installed. Look at README.md in repo 'LightAutoML' for installation instructions.
'nlp' extra dependency package 'transformers' isn't installed. Look at README.md in repo 'LightAutoML' for installation instructions.


In [95]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

# AutoWoE из LightAutoML
#from lightautoml.addons.autowoe import AutoWoE
from autowoe import AutoWoE, ReportDeco

### ---- Stability functions

In [37]:
import os
import warnings
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

# Настройка логгирования
#logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

LOG_FILE = "stability_tags.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE, mode="a", encoding="utf-8"),
        logging.StreamHandler(),
    ],
    force=True,
)

def analyze_feature_stability_optimized_prev(df, feature_cols, date_col, save_plots = True):
    """
    Оптимизированный анализ стабильности признаков во времени.

    Параметры
    ----------
    df : pd.DataFrame
        Исходный датафрейм.
    feature_cols : list[str]
        Список колонок-признаков (например, TAG_*).
    date_col : str
        Название колонки с датой.

    Возвращает
    ----------
    dict
        {
            'fill_ratio_by_month': DataFrame - заполненность по месяцам,
            'mode_match_ratio_by_month': DataFrame - доля совпадений с модой по месяцам,
            'fill_stats': DataFrame - статистики по заполненности,
            'mode_stats': DataFrame - статистики по совпадению с модой,
            'stability_stats': DataFrame - объединённые статистики,
            'unstable_features': list[str] - список нестабильных признаков
        }
    """
    warnings.filterwarnings("ignore")
    logging.info("Начало анализа стабильности признаков")

    # --- 1. Подготовка данных -------------------------------------------------
    logging.info("Подготовка данных: приведение даты и фильтрация по дате, заполнение NaN нулями")
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])

    # Фильтр по дате (по аналогии с исходным кодом)
    #df = df[df[date_col] >= "2023-06-01"]
    df["year_month"] = df[date_col].dt.to_period("M")

    # 2. Заменяем все пропуски на 0
    df_filled = df.copy()
    df_filled[feature_cols] = df_filled[feature_cols].fillna(0)

    # 3. Группировка по месяцам
    gb = df_filled.groupby("year_month")

    # 4. Заполненность = доля != 0
    logging.info("Расчет заполненности по месяцам")
    fill_ratio_by_month = gb[feature_cols].apply(lambda x: (x != 0).mean())

    # --- 5. Глобальная мода для каждой фичи (игнорируем 0) -------------------
    logging.info("Поиск глобальной моды для каждой фичи (игнорируя 0)")
    df_nonzero = df[feature_cols].replace(0, pd.NA).dropna(how="all")
    if not df_nonzero.empty:
        global_mode = df_nonzero.mode(dropna=True).iloc[0]
    else:
        global_mode = pd.Series(dtype=object)

    # Если мода не найдена для какого-то признака — исключаем его из расчёта
    valid_modes = global_mode.dropna()
    logging.info(f"Найдены моды для {len(valid_modes)} фич из {len(feature_cols)}")

    # --- 6. Доля совпадений с модой по месяцам --------------------------------
    logging.info("Расчет доли совпадений с модой по месяцам")
    if len(valid_modes) == 0:
        mode_match_ratio_by_month = pd.DataFrame(
            index=fill_ratio_by_month.index,
            columns=feature_cols,
            dtype=float,
        )
    else:
        subset_cols = valid_modes.index.tolist()
        mode_vals = valid_modes.to_dict()

        match_df = df_filled[["year_month"] + subset_cols].copy()
        for col in tqdm(subset_cols, desc="Расчет совпадений с модой"):
            match_df[col] = (match_df[col] == mode_vals[col]).astype(int)

        mode_match_by_month = match_df.groupby("year_month")[subset_cols].mean()

        mode_match_ratio_by_month = pd.DataFrame(
            index=fill_ratio_by_month.index,
            columns=feature_cols,
            dtype=float,
        )
        mode_match_ratio_by_month[subset_cols] = mode_match_by_month.reindex(fill_ratio_by_month.index)

    # --- 7. Статистика по заполненности ---------------------------------------
    logging.info("Расчет статистики заполненности")
    fill_stats = pd.DataFrame({
        "min_fill": fill_ratio_by_month.min(),
        "max_fill": fill_ratio_by_month.max(),
        "var_fill": fill_ratio_by_month.var(),
    })

    # --- 8. Статистика по совпадениям с модой ---------------------------------
    logging.info("Расчет статистики доли совпадений с модой")
    mode_stats = pd.DataFrame({
        "min_mode_match": mode_match_ratio_by_month.min(),
        "max_mode_match": mode_match_ratio_by_month.max(),
        "var_mode": mode_match_ratio_by_month.var(),
    })

    # --- 9. Объединённая таблица ----------------------------------------------
    stability_stats = fill_stats.join(mode_stats, how="outer")

    # --- 10. Адаптивные пороги дисперсии (95-й перцентиль) --------------------
    var_fill_clean = fill_stats["var_fill"].dropna()
    var_mode_clean = mode_stats["var_mode"].dropna()

    var_threshold_fill = var_fill_clean.quantile(0.95) if not var_fill_clean.empty else 0.0
    var_threshold_mode = var_mode_clean.quantile(0.95) if not var_mode_clean.empty else 0.0

    logging.info(f"Порог дисперсии заполненности (95%): {var_threshold_fill:.4f}")
    logging.info(f"Порог дисперсии совпадений с модой (95%): {var_threshold_mode:.4f}")

    # --- 11. Относительные изменения между месяцами ---------------------------
    fill_changes_rel = fill_ratio_by_month.pct_change().abs()
    fill_changes_rel = fill_changes_rel.replace([np.inf, -np.inf], np.nan)

    mode_changes_rel = mode_match_ratio_by_month.pct_change().abs()
    mode_changes_rel = mode_changes_rel.replace([np.inf, -np.inf], np.nan)

    # --- 12. Квантили: 2.5% и 97.5% ------------------------------------------
    q25_fill = fill_ratio_by_month.quantile(0.025, axis=0)
    q975_fill = fill_ratio_by_month.quantile(0.975, axis=0)

    q25_mode = mode_match_ratio_by_month.quantile(0.025, axis=0)
    q975_mode = mode_match_ratio_by_month.quantile(0.975, axis=0)

    # --- 13. Относительная разница (max - min) / min --------------------------
    min_fill = fill_ratio_by_month.min()
    max_fill = fill_ratio_by_month.max()
    rel_diff_fill = np.where(
        min_fill > 0,
        (max_fill - min_fill) / min_fill,
        np.where(max_fill > 0, np.inf, 0),
    )
    rel_diff_fill = pd.Series(rel_diff_fill, index=min_fill.index)

    min_mode = mode_match_ratio_by_month.min()
    max_mode = mode_match_ratio_by_month.max()
    rel_diff_mode = np.where(
        min_mode > 0,
        (max_mode - min_mode) / min_mode,
        np.where(max_mode > 0, np.inf, 0),
    )
    rel_diff_mode = pd.Series(rel_diff_mode, index=min_mode.index)

    rel_diff_threshold = 10  # оставляю ваш порог, комментарий можно поправить при желании
    unstable_by_diff_fill = set(rel_diff_fill[rel_diff_fill > rel_diff_threshold].index)
    unstable_by_diff_mode = set(rel_diff_mode[rel_diff_mode > rel_diff_threshold * 2].index)

    # --- 14. Абсолютный критерий: средняя заполненность < 0.01 ----------------
    mean_fill = fill_ratio_by_month.mean()
    unstable_by_critical_low_fill = set(mean_fill[mean_fill < 0.001].index)

    # --- 15. Критерий: низкая заполненность до 2025 года и высокая в 2025 -----
    logging.info("Поиск признаков с низкой заполненностью до 2025 года и высокой в 2025 году")

    fill_before_2025 = fill_ratio_by_month[fill_ratio_by_month.index.year < 2025]
    fill_in_2025 = fill_ratio_by_month[fill_ratio_by_month.index.year == 2025]

    mean_fill_before_2025 = fill_before_2025.mean()
    mean_fill_in_2025 = fill_in_2025.mean()

    low_fill_before_2025 = mean_fill_before_2025[mean_fill_before_2025 < 0.1].index
    high_fill_in_2025 = mean_fill_in_2025[mean_fill_in_2025 >= 0.1].index

    # Признаки, которые были низко заполнены до 2025, но высоко заполнены в 2025
    stable_by_fill_change = set(low_fill_before_2025).intersection(high_fill_in_2025)

    # Исключаем их из списка нестабильных по критически низкой заполненности
    unstable_by_critical_low_fill = unstable_by_critical_low_fill - stable_by_fill_change

    # --- 16. Дисперсия --------------------------------------------------------
    unstable_by_var_fill = set(stability_stats[stability_stats["var_fill"] > var_threshold_fill].index)
    unstable_by_var_mode = set(stability_stats[stability_stats["var_mode"] > var_threshold_mode].index)

    # --- 17. Относительные изменения (более 3 месяцев) ------------------------
    n_rel_anom_fill = (fill_changes_rel > 0.99).sum()
    n_rel_anom_mode = (mode_changes_rel > 0.99).sum()
    unstable_by_rel_change_fill = set(n_rel_anom_fill[n_rel_anom_fill > 2].index)
    unstable_by_rel_change_mode = set(n_rel_anom_mode[n_rel_anom_mode > 2].index)

    # --- 18. Экстремумы (<2.5% или >97.5%, более 2 месяцев) -------------------
    n_low_fill = (fill_ratio_by_month < q25_fill).sum()
    n_high_fill = (fill_ratio_by_month > q975_fill).sum()
    n_low_mode = (mode_match_ratio_by_month < q25_mode).sum()
    n_high_mode = (mode_match_ratio_by_month > q975_mode).sum()

    unstable_by_extreme_fill_low = set(n_low_fill[n_low_fill > 2].index)
    unstable_by_extreme_fill_high = set(n_high_fill[n_high_fill > 2].index)
    unstable_by_extreme_mode_low = set(n_low_mode[n_low_mode > 2].index)
    unstable_by_extreme_mode_high = set(n_high_mode[n_high_mode > 2].index)

    # --- 19. Фильтр «признак отключён в 2025» ---------------------------------
    logging.info("Поиск признаков, отключённых в 2025 г. (>6 месяцев с 0 < fill < 0.001)")
    fill_2025 = fill_ratio_by_month[fill_ratio_by_month.index.year == 2025]
    n_near_zero_2025 = ((fill_2025 > 0) & (fill_2025 < 0.001)).sum()
    switched_off_2025 = set(n_near_zero_2025[n_near_zero_2025 > 6].index)

    # --- 20. Объединение всех критериев ---------------------------------------
    unstable_features = sorted(list(
        unstable_by_critical_low_fill |
        switched_off_2025 |
        unstable_by_var_fill |
        unstable_by_var_mode |
        unstable_by_rel_change_fill |
        unstable_by_rel_change_mode |
        unstable_by_extreme_fill_low |
        unstable_by_extreme_fill_high |
        unstable_by_extreme_mode_low |
        unstable_by_extreme_mode_high 
    ))

    stable_features = list(set(set(feature_cols) - set(unstable_features))) ### ADJUST

    # --- 21. Логгирование ------------------------------------------------------
    logging.info(f"Найдено {len(unstable_features)} нестабильных фич:")
    logging.info(f"  • fill < 0.001 (среднее): {len(unstable_by_critical_low_fill)}")
    logging.info(f"  • fill ~ 0 (отключены в 2025): {len(switched_off_2025)}")
    logging.info(f"  • var_fill > 95% перцентиля: {len(unstable_by_var_fill)}")
    logging.info(f"  • var_mode > 95% перцентиля: {len(unstable_by_var_mode)}")
    logging.info(f"  • rel_change_fill > 99% (более 3 мес): {len(unstable_by_rel_change_fill)}")
    logging.info(f"  • rel_change_mode > 99% (более 3 мес): {len(unstable_by_rel_change_mode)}")
    logging.info(f"  • extreme_fill_low (<2.5%, более 2 мес): {len(unstable_by_extreme_fill_low)}")
    logging.info(f"  • extreme_fill_high (>97.5%, более 2 мес): {len(unstable_by_extreme_fill_high)}")
    logging.info(f"  • extreme_mode_low (<2.5%, более 2 мес): {len(unstable_by_extreme_mode_low)}")
    logging.info(f"  • extreme_mode_high (>97.5%, более 2 мес): {len(unstable_by_extreme_mode_high)}")
    

    # --- 22. Построение и сохранение графиков ---------------------------------
    if save_plots:
        logging.info("Построение и сохранение графиков для нестабильных фич")

        output_dir = "FEATURES_STABILITY_UNSTABLE"
        os.makedirs(output_dir, exist_ok=True)

        x_dates = fill_ratio_by_month.index.to_timestamp()

        for feat in unstable_features:
            fig, ax = plt.subplots(figsize=(10, 5))

            ax.plot(x_dates, fill_ratio_by_month[feat].values,
                    label="Заполненность (≠0)", marker="o")
            ax.plot(x_dates, mode_match_ratio_by_month[feat].values,
                    label="Совпадение с модой", marker="x")

            ax.set_title(f"Фича: {feat}")
            ax.set_ylabel("Доля")
            ax.set_xlabel("Месяц")
            ax.legend()
            fig.autofmt_xdate()
            fig.tight_layout()

            # безопасное имя файла
            safe_name = "".join(
                c if c.isalnum() or c in ("_", "-") else "_" for c in str(feat)
            )
            save_path = os.path.join(output_dir, f"{safe_name}.png")
            fig.savefig(save_path, dpi=200)
            plt.close(fig)

            logging.info(f"Сохранён график: {save_path}")

# Построение стабильных
        if save_plots:
            logging.info("Построение и сохранение графиков для стабильных фич")

            output_dir = "FEATURES_STABILITY_STABLE"
            os.makedirs(output_dir, exist_ok=True)

            x_dates = fill_ratio_by_month.index.to_timestamp()

            for feat in stable_features:
                fig, ax = plt.subplots(figsize=(10, 5))

                ax.plot(x_dates, fill_ratio_by_month[feat].values,
                        label="Заполненность (≠0)", marker="o")
                ax.plot(x_dates, mode_match_ratio_by_month[feat].values,
                        label="Совпадение с модой", marker="x")

                ax.set_title(f"Фича: {feat}")
                ax.set_ylabel("Доля")
                ax.set_xlabel("Месяц")
                ax.legend()
                fig.autofmt_xdate()
                fig.tight_layout()

                # безопасное имя файла
                safe_name = "".join(
                    c if c.isalnum() or c in ("_", "-") else "_" for c in str(feat)
                )
                save_path = os.path.join(output_dir, f"{safe_name}.png")
                fig.savefig(save_path, dpi=200)
                plt.close(fig)

                logging.info(f"Сохранён график: {save_path}")

    logging.info("Анализ завершен")

    return {
        "fill_ratio_by_month": fill_ratio_by_month,
        "mode_match_ratio_by_month": mode_match_ratio_by_month,
        "fill_stats": fill_stats,
        "mode_stats": mode_stats,
        "stability_stats": stability_stats,
        "unstable_features": unstable_features,
        "stable_features": stable_features,
    }


### ---- columns type

In [38]:
def get_columns_and_types(df_name, csv_prefix = None):
    columns_and_types = [(col, df_name[col].dtype.name) for col in df_name.columns]

    # Преобразуем этот список в DataFrame
    info_df = pd.DataFrame(columns_and_types, columns=['Column_name', 'Dtype'])
    if csv_prefix is not None:
        info_df.to_csv(f'{csv_prefix}_Columns_dtype_org.csv', sep=';', encoding = 'cp1251', decimal =',')

    # Группируем по типу данных и подсчитываем количество колонок каждого типа
    grouped = info_df.groupby('Dtype').size().reset_index(name='Columns_cnt')

    # Добавляем долю колонок для каждого типа данных
    total_columns = len(df_name.columns)
    grouped['Distr'] = grouped['Columns_cnt'] / total_columns * 100

    return grouped

### ---- HT NAMES CATALOG

In [39]:
HT_catalog_path = r"E:\PubPricing\01_Retail\03_PPI\01_Modelling\01_HT\00_HT_Descr\pyCatalog\HT_names_prom_2026.csv"
HT_catalog_csv = pd.read_csv(HT_catalog_path, low_memory=False, decimal ='.', encoding = 'utf-8', sep = ';')
tags_name_map = HT_catalog_csv.set_index('NAME_LAMA')['TAG_NAME'].to_dict()
# print(tags_name_map.get("TAG_14232"))

### ---- FSA - OW gini and mixed-type correlation

In [40]:
# ---------------------------
# ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ---------------------------

def _is_categorical(series: pd.Series, max_unique_ratio: float = 0.05, force_object_as_cat: bool = True) -> bool:
    """Эвристика: объектные/категориальные типы -> cat; числовые, но с очень малым числом уникальных значений -> cat."""
    if force_object_as_cat and (series.dtype == 'object' or str(series.dtype).startswith('category') or series.dtype == 'bool'):
        return True
    if np.issubdtype(series.dtype, np.number):
        nunique = series.nunique(dropna=True)
        return nunique <= max(10, int(len(series) * max_unique_ratio))
    return True

def _safe_numeric(s: pd.Series):
    """Преобразовать к float, заменить inf/−inf на NaN."""
    s = pd.to_numeric(s, errors='coerce')
    s = s.replace([np.inf, -np.inf], np.nan)
    return s

def _gini_normalized(y_true: np.ndarray, y_score: np.ndarray) -> float:
    """
    Нормализованный Gini по определению Kaggle:
    Gini(y, s) = 2*Lorenz(s) - 1, нормализуем на Gini(y, y).
    Здесь s — скор/ранг; для регрессии используем сам признак как скор.
    """
    # Уберём NaN синхронно
    mask = np.isfinite(y_true) & np.isfinite(y_score)
    y_true = np.asarray(y_true)[mask]
    y_score = np.asarray(y_score)[mask]
    if len(y_true) < 2:
        return np.nan

    def _gini(y, pred):
        # сортируем по pred по убыванию
        order = np.argsort(-pred, kind="mergesort")
        y = y[order]
        # накопленные суммы
        cum_y = np.cumsum(y)
        sum_y = cum_y[-1]
        if sum_y == 0:
            return 0.0
        # Лоренц-кривая на дискретной сетке
        lorenz = cum_y / sum_y
        n = len(y)
        g = (lorenz - (np.arange(1, n + 1) / n)).sum()
        return g / n

    g = _gini(y_true, y_score)
    g_perfect = _gini(y_true, y_true)  # нормирующая константа
    return np.nan if g_perfect == 0 else g / g_perfect

def _cramers_v(x: pd.Series, y: pd.Series) -> float:
    """Cramér’s V для двух категориальных признаков (с поправкой на размер таблицы)."""
    tbl = pd.crosstab(x, y)
    if tbl.size == 0:
        return np.nan
    chi2, _, _, _ = chi2_contingency(tbl, correction=False)
    n = tbl.values.sum()
    if n == 0:
        return np.nan
    phi2 = chi2 / n
    r, k = tbl.shape
    # корректировка Байкла–Крамера
    phi2corr = max(0, phi2 - (k - 1)*(r - 1)/(n - 1))
    rcorr = r - (r - 1)**2 / (n - 1)
    kcorr = k - (k - 1)**2 / (n - 1)
    denom = min((kcorr - 1), (rcorr - 1))
    return np.sqrt(phi2corr / denom) if denom > 0 else np.nan

def _correlation_ratio(categories: pd.Series, values: pd.Series) -> float:
    """Корреляционное отношение η(num ~ cat). categories — категориальная, values — числовая."""
    values = _safe_numeric(values)
    mask = categories.notna() & values.notna()
    if mask.sum() < 2:
        return np.nan
    categories = categories[mask]
    values = values[mask]

    groups = []
    for k, grp in values.groupby(categories):
        if len(grp) > 0:
            groups.append(grp.values)

    if len(groups) < 2:
        return np.nan
    all_vals = values.values
    y_mean = all_vals.mean()
    ss_between = sum([len(g) * (g.mean() - y_mean)**2 for g in groups])
    ss_total = ((all_vals - y_mean)**2).sum()
    return np.sqrt(ss_between / ss_total) if ss_total > 0 else np.nan

# ---------------------------
# 1) ОДНОФАКТОРНЫЙ GINI
# ---------------------------

def one_factor_gini_classification(df: pd.DataFrame, target: str) -> pd.DataFrame:
    """
    Для бинарной классификации: Gini = 2*AUC - 1,
    где в качестве score берём сам признак (для категориальных — среднее таргета по категориям).
    """
    y = df[target].values
    if set(np.unique(y)).issubset({0, 1}) is False:
        raise ValueError("Целевой признак должен быть бинарным (0/1).")

    out = []
    for col in tqdm(df.columns, desc="One-factor GINI (Classification)"):
        if col == target:
            continue
        s = df[col]
        # обработка по типу
        if _is_categorical(s):
            # target-mean encoding (simple, без кросс-валидации для one-factor оценки)
            means = df.groupby(col, dropna=False)[target].mean()
            score = s.map(means)
        else:
            score = _safe_numeric(s)

        # AUC может упасть, если все NaN или константа
        score = score.astype(float)
        # Импутация NaN медианой (иначе roc_auc_score упадёт)
        if score.notna().sum() == 0 or score.nunique(dropna=True) < 2:
            gini = np.nan
        else:
            filled = score.fillna(score.median())
            try:
                auc = roc_auc_score(y, filled)
                gini = 2 * auc - 1
            except ValueError:
                gini = np.nan

        out.append((col, gini))
    res = pd.DataFrame(out, columns=["feature", "gini"]).sort_values("gini", ascending=False, na_position="last")
    return res.reset_index(drop=True)

def one_factor_gini_regression(df: pd.DataFrame, target: str) -> pd.DataFrame:
    """
    Для регрессии: нормализованный Gini между y и «скором»,
    где скором является сам признак (для категориальных — среднее целевого по категориям).
    Это мера ранговой дискр.способности признака относительно y.
    """
    y = _safe_numeric(df[target]).values
    out = []
    for col in tqdm(df.columns, desc="One-factor GINI (Regression)"):
        if col == target:
            continue
        s = df[col]
        if _is_categorical(s):
            means = df.groupby(col, dropna=False)[target].mean()
            score = s.map(means)
        else:
            score = _safe_numeric(s)

        if pd.isna(score).all() or score.nunique(dropna=True) < 2:
            g = np.nan
        else:
            # Импутация медианой, чтобы удалять минимум наблюдений
            filled = score.fillna(score.median()).values
            g = _gini_normalized(y_true=y, y_score=filled)
        out.append((col, g))
    res = pd.DataFrame(out, columns=["feature", "gini_normalized"]).sort_values("gini_normalized", ascending=False, na_position="last")
    return res.reset_index(drop=True)

# ---------------------------
# 2) МАТРИЦА ПОПАРНЫХ КОРРЕЛЯЦИЙ ДЛЯ СМЕШАННЫХ ТИПОВ
# ---------------------------

def mixed_correlation_matrix(df: pd.DataFrame, prefer_spearman: bool = False) -> pd.DataFrame:
    """
    Возвращает квадратную матрицу «смешанной корреляции»:
    - num–num: Pearson (или Spearman, если prefer_spearman=True)
    - cat–cat: Cramér’s V
    - num–cat: η (correlation ratio)
    """
    cols = df.columns.tolist()
    types = {c: ('cat' if _is_categorical(df[c]) else 'num') for c in cols}
    n = len(cols)
    mat = np.zeros((n, n), dtype=float)
    mat[:] = np.nan

    # Предрасчёт числовых и категориальных представлений
    num_data = {c: _safe_numeric(df[c]) for c in cols}
    cat_data = {c: df[c].astype('object') for c in cols}

    # Быстрый блок для num-num (симметрично)
    num_cols = [c for c in cols if types[c] == 'num']
    if prefer_spearman and len(num_cols) >= 2:
        # Спирмен
        ranks = pd.DataFrame({c: num_data[c].rank(method='average') for c in num_cols})
        pear = ranks.corr(method='pearson', min_periods=2)
        for i, ci in enumerate(num_cols):
            for j, cj in enumerate(num_cols):
                mat[cols.index(ci), cols.index(cj)] = pear.iloc[i, j]
    elif len(num_cols) >= 2:
        pear = pd.DataFrame({c: num_data[c] for c in num_cols}).corr(method='pearson', min_periods=2)
        for i, ci in enumerate(num_cols):
            for j, cj in enumerate(num_cols):
                mat[cols.index(ci), cols.index(cj)] = pear.iloc[i, j]
    elif len(num_cols) == 1:
        mat[cols.index(num_cols[0]), cols.index(num_cols[0])] = 1.0

    # Остальные пары
    for i, ci in tqdm(enumerate(cols), total=len(cols), desc="Mixed Correlation Matrix"):
        for j, cj in enumerate(cols):
            if i == j:
                mat[i, j] = 1.0
                continue
            if not np.isnan(mat[i, j]):
                continue  # уже заполнено блоком num-num
            ti, tj = types[ci], types[cj]
            if ti == 'cat' and tj == 'cat':
                r = _cramers_v(cat_data[ci], cat_data[cj])
            elif ti == 'num' and tj == 'cat':
                r = _correlation_ratio(cat_data[cj], num_data[ci])
            elif ti == 'cat' and tj == 'num':
                r = _correlation_ratio(cat_data[ci], num_data[cj])
            else:
                # этот случай закрыт выше, но на всякий
                r = pd.concat([num_data[ci], num_data[cj]], axis=1).corr(method='pearson', min_periods=2).iloc[0, 1]
            mat[i, j] = r

    return pd.DataFrame(mat, index=cols, columns=cols)

In [42]:
# EG of use
if False:
    print("\n=== One-factor GINI (classification) ===")
    print(one_factor_gini_classification(df, target="y_bin"))

    print("\n=== One-factor normalized GINI (regression) ===")
    print(one_factor_gini_regression(df, target="y_reg"))

    print("\n=== Mixed-type correlation matrix (Pearson for num–num) ===")
    corr = mixed_correlation_matrix(df.drop(columns=["y_bin","y_reg"]), prefer_spearman=False)
    print(corr.round(3))

### ---- GINI Time Trend

#### ------ Import и функции расчета GINI и ДИ

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from scipy.stats import norm
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")


def gini_binary(y_true, y_pred):
    """Считает Gini через AUC"""
    auc = roc_auc_score(y_true, y_pred)
    return 2 * auc - 1


def gini_customized(y_true, y_pred, gini_normalized_bool=False):
    """Имя оставлено для совместимости со старыми вызовами ноутбука."""
    return gini_binary(y_true, y_pred)


def compute_confidence_interval(gini, n, alpha=0.05):
    """Простой нормальный аппрокс. ДИ"""
    se = np.sqrt((gini * (1 - gini)) / n) if n > 0 else 0
    z = norm.ppf(1 - alpha / 2)
    return gini - z * se, gini + z * se


#### ------ расчет динамики GINI: возвращает pd DateFrame

In [ ]:
def calc_gini_dynamics(df1, observed_col, predicted_col, date_col,
                       freq='M', min_observations=10,
                       alpha_95=0.05, alpha_99=0.01,
                       gini_normalized_bool=False):
    """
    Динамика Gini с доверительными интервалами.
    Gini считается как в gini_timetrend.py: 2 * roc_auc_score(y_true, y_pred) - 1.
    """

    freq = str(freq).upper()

    data = df1.copy()
    data[date_col] = pd.to_datetime(data[date_col])

    freq_map = {
        'D': 'D',
        'W': 'W',
        'M': 'M',
        'Q': 'Q',
        'Y': 'Y'
    }

    period_freq = freq_map.get(freq, 'M')
    grouped = data.groupby(data[date_col].dt.to_period(period_freq), sort=True)

    results = []

    for period, group_data in grouped:
        if len(group_data) >= min_observations:
            try:
                gini_model = gini_binary(group_data[observed_col], group_data[predicted_col])
                n_obs = len(group_data)

                ci_95_low, ci_95_high = compute_confidence_interval(gini_model, n_obs, alpha_95)
                ci_99_low, ci_99_high = compute_confidence_interval(gini_model, n_obs, alpha_99)

                actual = group_data[observed_col].mean()
                fitted = group_data[predicted_col].mean()
                diff_to_pred = actual / fitted

                results.append({
                    "period": period,
                    "gini": gini_model,
                    "n_observations": n_obs,
                    "ci_95_low": ci_95_low,
                    "ci_95_high": ci_95_high,
                    "ci_99_low": ci_99_low,
                    "ci_99_high": ci_99_high,
                    "value_actual": actual,
                    "value_fitted": fitted,
                    "value_parity_diff_to_pred": diff_to_pred,
                })
            except Exception as e:
                print(f"Ошибка в периоде {period}: {e}")
                continue

    if not results:
        print("Нет данных для графика")
        return None

    results_df1 = pd.DataFrame(results).sort_values('period')
    return results_df1


#### ------ построение графика динамики GINI

In [ ]:
def plot_gini_dynamics(df1, observed_col, predicted_col, date_col,
                       freq='M', min_observations=10,
                       alpha_95=0.05, alpha_99=0.01,
                       figsize=(14, 8), title=None,
                       gini_normalized_bool=False):
    """
    График динамики Gini.
    Расчет Gini внутри такой же, как в gini_timetrend.py.
    """

    freq = str(freq).upper()

    data = df1.copy()
    data[date_col] = pd.to_datetime(data[date_col])

    results_df1 = calc_gini_dynamics(
        df1, observed_col, predicted_col, date_col,
        freq, min_observations, alpha_95, alpha_99,
        gini_normalized_bool
    )

    if results_df1 is None or results_df1.empty:
        return None, None

    x_positions = np.arange(len(results_df1))

    fig, ax1 = plt.subplots(figsize=figsize)
    ax2 = ax1.twinx()

    # Столбцы = число наблюдений
    ax2.bar(x_positions, results_df1['n_observations'],
            alpha=0.3, color="#2ca02c", label="Количество наблюдений", width=0.6)

    # 99% ДИ
    ax1.fill_between(x_positions, results_df1['ci_99_low'], results_df1['ci_99_high'],
                     alpha=0.3, color="#ff4444", label="99% ДИ")

    # 95% ДИ
    ax1.fill_between(x_positions, results_df1['ci_95_low'], results_df1['ci_95_high'],
                     alpha=0.5, color="#ffcc00", label="95% ДИ")

    # Линия Gini
    ax1.plot(x_positions, results_df1['gini'], color='black',
             linewidth=2, marker='o', markersize=6, label="Gini")

    # подписи на точках
    for x_pos, (_, row) in zip(x_positions, results_df1.iterrows()):
        ax1.annotate(f"{row['gini']:.3f}",
                     (x_pos, row['gini']),
                     textcoords="offset points", xytext=(0, 10),
                     ha='center', fontsize=9, fontweight='bold')

    # Пороговые линии
    ax1.axhline(y=gini_binary(data[observed_col], data[predicted_col]), color='purple', linestyle='--', alpha=0.7, label='Среднее Gini')
    ax1.axhline(y=0.2, color='red', linestyle='--', alpha=0.7, label='Порог 20%')
    ax1.axhline(y=0.3, color='green', linestyle='--', alpha=0.7, label='Порог 30%')

    # Оси
    ax1.set_xlabel("Период", fontsize=12, fontweight='bold')
    ax1.set_ylabel("Gini", fontsize=12, fontweight='bold', color="#1f77b4")
    ax2.set_ylabel("Количество наблюдений", fontsize=12, fontweight='bold', color="#2ca02c")

    ax1.tick_params(axis='y', labelcolor="#1f77b4")
    ax2.tick_params(axis='y', labelcolor="#2ca02c")

    # Формат периодов без визуального сдвига к следующему месяцу/кварталу
    def _format_period(period):
        if freq == 'Q':
            return f"{period.year}-Q{period.quarter}"
        if freq == 'Y':
            return f"{period.year}"
        if freq == 'M':
            return period.start_time.strftime('%Y-%m')
        if freq == 'W':
            return f"{period.start_time:%Y-%m-%d}\n{period.end_time:%Y-%m-%d}"
        return period.start_time.strftime('%Y-%m-%d')

    ax1.set_xticks(x_positions)
    ax1.set_xticklabels([_format_period(period) for period in results_df1['period']])
    ax1.set_xlim(-0.5, len(results_df1) - 0.5)

    plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')

    if title is None:
        title = "Динамика Gini (бинарная классификация)"
    ax1.set_title(title, fontsize=14, fontweight='bold', pad=20)

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2,
               loc='upper left', frameon=True, fancybox=True, shadow=True)

    ax1.grid(True, alpha=0.3)
    plt.tight_layout()

    print(f"Построен график для {len(results_df1)} периодов")
    print(f"Gini: {gini_binary(data[observed_col], data[predicted_col]):.3f}")
    print(f"Мин Gini: {results_df1['gini'].min():.3f}")
    print(f"Макс Gini: {results_df1['gini'].max():.3f}")
    print(f"Среднее кол-во наблюдений: {results_df1['n_observations'].mean():.0f}")

    return fig, ax1


### --- Upload DB

In [2]:
main_db_path = r"E:\PubPricing\01_Retail\03_PPI\01_Modelling\01_HT\2026m03\00_DB\radar_property_w_tags_ness_fields_prom_v5_2026_01_not_mort_pt_23_25.csv"

HH_main_df = pd.read_csv(
    main_db_path,
    low_memory=False,
    sep='^',
    encoding='cp1251',
    decimal=','
)
HH_main_df.set_index('CO_KEY', inplace=True)

print(HH_main_df.shape)
HH_main_df.head()

(3197955, 778)


,CONTRACT_ID,OBJECT_TYPE,POLICY_DATE,POLICY_START_DATE,POLICY_END_DATE,POLICY_CANCEL_DATE,EXPOSURE_POLICY_PERIOD,ACC_CNT_TOTAL,ACC_CNT_FIRE,ACC_CNT_WATER,...,TAG_39247,TAG_39248,MULTI_CONTRACT_IND,INIT_CONTRACT_NUMBER,INIT_POLICY_START_DATE,INIT_POLICY_END_DATE,INIT_POLICY_CANCEL_DATE,DIFF_TO_INIT_START_DATE_DAYS,DIFF_TO_INIT_START_DATE_MONTHS,DIFF_TO_INIT_START_DATE_YEARS
CO_KEY,,,,,,,,,,,,,,,,,,,,,
9AE50050560121AD11EE996DB6CC239BКвартира,9AE50050560121AD11EE996DB6CC239B,Квартира,2023-12-13 00:00:00.0,2023-12-18 00:00:00.0,2024-01-17 00:00:00.0,NaN,0.084932,0,0,0,...,0,0,1,007WS5654972327,2023-08-18 00:00:00.0,2023-09-17 00:00:00.0,NaN,122,4,0
8289005056012C8811EDB9B8F3045F35Квартира,8289005056012C8811EDB9B8F3045F35,Квартира,2023-03-03 00:00:00.0,2023-03-11 00:00:00.0,2024-03-10 00:00:00.0,NaN,1.000000,0,0,0,...,0,0,0,NaN,2023-03-11 00:00:00.0,2024-03-10 00:00:00.0,NaN,0,0,0
8207005056012C8811ED0C3CA0BDF684Квартира,8207005056012C8811ED0C3CA0BDF684,Квартира,2022-07-25 00:00:00.0,2022-08-08 00:00:00.0,2023-08-07 00:00:00.0,NaN,1.000000,0,0,0,...,0,0,0,NaN,2022-08-08 00:00:00.0,2023-08-07 00:00:00.0,NaN,0,0,0
9B0A0050560A61CE11EF64EC36DCAF9CКвартира,9B0A0050560A61CE11EF64EC36DCAF9C,Квартира,2024-08-27 00:00:00.0,2024-08-28 00:00:00.0,2025-08-27 00:00:00.0,NaN,1.000000,0,0,0,...,0,0,0,NaN,2024-08-28 00:00:00.0,2025-08-27 00:00:00.0,NaN,0,0,0
9C080050560121AD11F0508E5149A42EКвартира,9C080050560121AD11F0508E5149A42E,Квартира,2025-06-24 00:00:00.0,2025-07-01 00:00:00.0,2025-07-31 00:00:00.0,NaN,0.084932,0,0,0,...,0,0,1,007WS4700246911,2025-01-29 00:00:00.0,2025-02-28 00:00:00.0,NaN,153,5,0


### --- Target & Time setup

In [3]:
TARGET_NAME       = 'ACC_CNT_WATER_BOOL'
TIME_TREND_FACTOR = 'POLICY_END_DATE'

HH_main_df[TIME_TREND_FACTOR] = pd.to_datetime(HH_main_df[TIME_TREND_FACTOR])

print('Target rate:', HH_main_df[TARGET_NAME].mean().round(4))
print('Date range: ', HH_main_df[TIME_TREND_FACTOR].min(), '→', HH_main_df[TIME_TREND_FACTOR].max())

Target rate: 0.0069
Date range:  2023-01-01 00:00:00 → 2025-12-31 00:00:00


### --- Train / Test split

In [4]:
RANDOM_STATE = 28
TEST_SIZE    = 0.4

df_train, df_test = train_test_split(
    HH_main_df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=HH_main_df[TARGET_NAME]
)

print(f'Train: {df_train.shape},  Test: {df_test.shape}')
print(f'Target rate — Train: {df_train[TARGET_NAME].mean():.4f},  Test: {df_test[TARGET_NAME].mean():.4f}')

Train: (1918773, 778),  Test: (1279182, 778)
Target rate — Train: 0.0069,  Test: 0.0069


# 01. Feature short list

In [6]:
# Загружаем short list признаков (аналогично оригинальному ноутбуку)
FeatShortList = list(set(
    pd.read_csv(
        r'FSA_FeatTruncList02.csv',
        low_memory=False, sep=';', encoding='cp1251', decimal=','
    )['NAME']
))

# Дополнительные продуктовые признаки
feat_list_internal = [
    'PRODUCT_OBJECT_TYPE_HOUSE',
    'PRODUCT_OBJECT_TYPE_FLAT',
    'PRODUCT_OBJECT_TYPE_OTHER',
]
FeatShortList += feat_list_internal

# Оставляем только те, что реально есть в данных
FeatShortList = [f for f in FeatShortList if f in df_train.columns]
print(f'Признаков в short list: {len(FeatShortList)}')

Признаков в short list: 363


# 02. AutoWoE — обучение модели

### Параметры AutoWoE

| Параметр | Описание |
|---|---|
| `monotonic` | Монотонность биннинга (True = монотонный WoE) |
| `max_bin_count` | Макс. число бинов на признак |
| `pearson_th` | Порог корреляции Пирсона для отбора (дропаем коррелирующие) |
| `auc_th` | Минимальный AUC признака для включения в модель |
| `vif_th` | Порог VIF (Variance Inflation Factor) |
| `imp_th` | Порог важности признака |
| `regularized_refit` | Регуляризованное переобучение финальной LR |

### --- Типы признаков

In [7]:
# AutoWoE требует явного указания типов: 'real' или 'cat'
# По умолчанию числовые -> 'real', object -> 'cat'

features_type = {}
for col in FeatShortList:
    if df_train[col].dtype == object or df_train[col].nunique() <= 10:
        features_type[col] = 'cat'
    else:
        features_type[col] = 'real'

print('real:', sum(v == 'real' for v in features_type.values()),
      '| cat:', sum(v == 'cat' for v in features_type.values()))

real: 195 | cat: 168


### --- Инициализация и обучение

In [96]:
autowoe = AutoWoE(
    monotonic=False,        # без принудительной монотонности
    max_bin_count=5,        # не более 5 бинов на признак
    select_type=None,       # отбор на основе AUC/IV
    pearson_th=0.9,         # дропаем если корреляция > 0.9
    auc_th=0.505,           # минимальный AUC для включения
    vif_th=10.0,            # VIF порог
    imp_th=0,               # порог важности
    target_name=TARGET_NAME,
    regularized_refit=False,
    p_val=0.05,             # p-value для отбора в LR
    verbose=2,
)
autowoe = ReportDeco(autowoe, )

--- Logging error ---
Traceback (most recent call last):
  File "E:\Python312\Lib\logging\__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "E:\Python312\Lib\logging\__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "E:\Python312\Lib\logging\__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "E:\Python312\Lib\logging\__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "e:\Python312\lightautoml_venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "e:\Python312\lightautoml_venv\Lib\site-packages\traitlets\config\application.py", line 1075, in la

In [97]:
%%time
autowoe.fit(
    df_train[FeatShortList + [TARGET_NAME]],
    target_name=TARGET_NAME,
    features_type=features_type,
)

 features ['PRODUCT_OBJECT_TYPE_OTHER'] contain too many nans or identical values
[LightGBM] [Info] Number of positive: 10474, number of negative: 1524544
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.677567 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 34003
[LightGBM] [Info] Number of data points in the train set: 1535018, number of used features: 362
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.006823 -> initscore=-4.980555
[LightGBM] [Info] Start training from score -4.980555
Training until validation scores don't improve for 10 rounds
Early stopping, best iteration is:
[33]	val_test's auc: 0.698989
 features ['TAG_11838', 'TAG_34204', 'TAG_33562', 'TAG_33538', 'TAG_10245', 'TAG_39039', 'TAG_10094', 'TAG_21678', 'TAG_39414', 'TAG_26562', 'TAG_14244', 'TAG_11847', 'TAG_33170', 'TAG_10421', 'TAG_11496', 'TAG_14253', 'TAG_3354

# 03. Predict & Метрики

### --- Предсказание

In [98]:
train_prediction = autowoe.model.predict_proba(df_train)

In [99]:
test_prediction = autowoe.predict_proba(df_test)

In [100]:
print("GINI Train:\t", 2*roc_auc_score(df_train[TARGET_NAME].values, train_prediction)-1)
print("GINI Test:\t", 2*roc_auc_score(df_test[TARGET_NAME].values, test_prediction)-1)

GINI Train:	 0.3801648744819286
GINI Test:	 0.37176505750324274


In [101]:
df_train['Pred_fin_LAMA'] = train_prediction

In [102]:
df_test['Pred_fin_LAMA'] = test_prediction

In [103]:
if False:
    df_train[FeatShortList + [TIME_TREND_FACTOR, TARGET_NAME, 'Pred_fin_LAMA']].to_csv("df_train_scored.csv", sep=';', encoding = 'cp1251', decimal =',')
    df_test[FeatShortList + [TIME_TREND_FACTOR, TARGET_NAME, 'Pred_fin_LAMA']].to_csv("df_test_scored.csv", sep=';', encoding = 'cp1251', decimal =',')

### --- Features list

In [104]:
model_represenation_dict = autowoe.get_model_represenation()

In [105]:
# Selected feats
selected_feats_binary = sorted(list(model_represenation_dict.get('features')))
print('LightAutoML AUTOWOE has selected {} feats!'.format(len(selected_feats_binary)))
for i in selected_feats_binary:
    print(i)

LightAutoML AUTOWOE has selected 20 feats!
PRODUCT_OBJECT_TYPE_FLAT
TAG_10281
TAG_10900
TAG_10935
TAG_10967
TAG_11489
TAG_12854
TAG_14235
TAG_14256
TAG_15943
TAG_20569
TAG_21903
TAG_22084
TAG_22832
TAG_24074
TAG_30327
TAG_33509
TAG_33577
TAG_34120
TAG_38778


In [106]:
SummaryStats_tbl = pd.DataFrame({
    'MODE': ['AUTOWOE'],
    'FEATURE_GROUP_SIZE': ['NA'],
    'MAX_TUNING_TIME_M': ['NA'],
    'METRIC': ['NA'],
    'CV_FOLDS': ['NA'],
    'TEST_SIZE': [TEST_SIZE],
    'USED_FEATS': len(sorted(list(model_represenation_dict.get('features')))),
    'TRAIN_GINI': [2*roc_auc_score(df_train[TARGET_NAME].values, train_prediction)-1],
    'OOF_GINI': ['NA'],
    'TEST_GINI': [2*roc_auc_score(df_test[TARGET_NAME].values, test_prediction)-1],
    'MODEL_STRUCTURE' : ['AUTOWOE']
})

In [107]:
SummaryStats_tbl

,MODE,FEATURE_GROUP_SIZE,MAX_TUNING_TIME_M,METRIC,CV_FOLDS,TEST_SIZE,USED_FEATS,TRAIN_GINI,OOF_GINI,TEST_GINI,MODEL_STRUCTURE
0,AUTOWOE,NA,NA,NA,NA,0.4,20,0.380165,NA,0.371765,AUTOWOE


In [108]:
SummaryStats_tbl.to_csv('SummaryStats_tbl.csv', sep=';', encoding = 'cp1251', decimal =',')

#### --- Make GINI Time Trend

#### -------- GINI TRAIN

In [109]:
oot_df1 = pd.DataFrame(df_train[TARGET_NAME])
oot_df1['FIN_pred'] = pd.DataFrame(df_train['Pred_fin_LAMA'])#oof_pred_binary.data[:, 0]
oot_df1['Date'] = df_train[TIME_TREND_FACTOR]

In [ ]:
plot_gini_dynamics(oot_df1, TARGET_NAME, "FIN_pred", "Date", freq = "M", title = "GINI TREND TRAIN", gini_normalized_bool=False)

##### --------- save to csv

In [ ]:
gini_time_trend_train = calc_gini_dynamics(oot_df1, TARGET_NAME, "FIN_pred", "Date", freq = "M", gini_normalized_bool=False)
gini_time_trend_train.to_csv('gini_time_trend_train.csv', sep=';', encoding = 'cp1251', decimal =',')

In [ ]:
gini_time_trend_train_q = calc_gini_dynamics(oot_df1, TARGET_NAME, "FIN_pred", "Date", freq = "Q", gini_normalized_bool=False)
gini_time_trend_train_q.to_csv('gini_time_trend_train_q.csv', sep=';', encoding = 'cp1251', decimal =',')

#### --------➡️ GINI TEST

In [ ]:
oot_df2 = pd.DataFrame(df_test[TARGET_NAME])
oot_df2['FIN_pred'] = pd.DataFrame(df_test['Pred_fin_LAMA'])#oof_pred_binary.data[:, 0]
oot_df2['Date'] = df_test[TIME_TREND_FACTOR]

plot_gini_dynamics(oot_df2, TARGET_NAME, "FIN_pred", "Date", freq = "M", title = "GINI TREND TEST", gini_normalized_bool=False)

gini_time_trend_test = calc_gini_dynamics(oot_df2, TARGET_NAME, "FIN_pred", "Date", freq = "M", gini_normalized_bool=False)
gini_time_trend_test.to_csv('gini_time_trend_test.csv', sep=';', encoding = 'cp1251', decimal =',')

gini_time_trend_test_q = calc_gini_dynamics(oot_df2, TARGET_NAME, "FIN_pred", "Date", freq = "Q", gini_normalized_bool=False)
gini_time_trend_test_q.to_csv('gini_time_trend_test_q.csv', sep=';', encoding = 'cp1251', decimal =',')

In [ ]:
gini_time_trend_test

In [ ]:
gini_time_trend_test_q

# 04. Интерпретация модели - Generate report

In [116]:
report_params = {"output_path": "HT_HH_PROB_WD_IFL_model_report", # folder for report generation
                 "report_name": "SBS #TAGS MODELLING NONE-MOTOR. WHITEBOX REPORT",
                 "report_version_id": 1,
                  "city": "Moscow",
                  "model_aim": "Probability of water damage in Household Insurance products",
                  "model_name": "HH HT WD PROB",
                  "zakazchik": "SBS",
                  "high_level_department": "SBS",
                  "ds_name": "SBS",
                  "target_descr": "water damage",
                  "non_target_descr": "no claims appear"
}

autowoe.generate_report(report_params, )

2026-05-13 02:08:21,267 - INFO - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-05-13 02:08:21,282 - INFO - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-05-13 02:08:23,702 - INFO - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-05-13 02:08:23,712 - INFO - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-05-13 02:08:25,994 - INFO - Using categorical units to plot a list of strings that are all parsable as 

Successfully wrote HT_HH_PROB_WD_IFL_model_report\autowoe_report.html.


# 05. Сохранение модели и результатов

### --- Сохранение модели

In [117]:
joblib.dump(autowoe.model, 'HT_HH_PROB_WD_AUTOWOE_model.pkl')
print('Модель сохранена: HT_HH_PROB_WD_AUTOWOE_model.pkl')

Модель сохранена: HT_HH_PROB_WD_AUTOWOE_model.pkl


### --- SQL

In [118]:
sql_query = autowoe.model.get_sql_inference_query('global_temp.TABLE_1')

In [119]:
print(sql_query)

SELECT
  1 / (1 + EXP(-(
    -4.982
    -0.991*WOE_TAB.PRODUCT_OBJECT_TYPE_FLAT
    -0.965*WOE_TAB.TAG_11489
    -0.524*WOE_TAB.TAG_22084
    -0.525*WOE_TAB.TAG_10900
    -0.527*WOE_TAB.TAG_20569
    -0.451*WOE_TAB.TAG_24074
    -0.478*WOE_TAB.TAG_34120
    -0.711*WOE_TAB.TAG_15943
    -0.525*WOE_TAB.TAG_21903
    -1.209*WOE_TAB.TAG_30327
    -0.626*WOE_TAB.TAG_10935
    -0.46*WOE_TAB.TAG_38778
    -0.331*WOE_TAB.TAG_14235
    -0.599*WOE_TAB.TAG_33577
    -0.398*WOE_TAB.TAG_14256
    -0.459*WOE_TAB.TAG_22832
    -0.493*WOE_TAB.TAG_10967
    -0.524*WOE_TAB.TAG_33509
    -0.457*WOE_TAB.TAG_12854
    -0.491*WOE_TAB.TAG_10281
  ))) as PROB,
  WOE_TAB.*
FROM 
    (SELECT
    CASE
      WHEN PRODUCT_OBJECT_TYPE_FLAT == 0 THEN 1.751
      WHEN PRODUCT_OBJECT_TYPE_FLAT == 1 THEN -0.245
      ELSE 0
    END AS PRODUCT_OBJECT_TYPE_FLAT,
    CASE
      WHEN (TAG_11489 IS NULL OR TAG_11489 = 'NaN') THEN -0.024
      WHEN TAG_11489 <= 0.705 THEN 0.884
      WHEN TAG_11489 <= 0.885 THEN 1.087
      

In [120]:
type(sql_query)

str

In [121]:
with open("sql_query.txt", "w", encoding="utf-8") as file: 
    file.write(sql_query)